# Motor de inferencia de Arritmias
##### En este archivo se detalla el paso a paso para el entrenamiento de una CNN1D compatible con un Ergometro portatil de una derivacion. Objetivo: obtener un modelo entrenado y validado para la deteccion de arritmias usando como BD de entrenamiento, testeo y validacion la MIT BIH de Physionet.

Paso 1) Adecuacion del entorno

In [ ]:
!pip install wfdb -q

In [ ]:
import wfdb
import os

#Uso wfdb para bajar los archivos del MITBIH de physionet
DATA_DIR = "mitdb"
os.makedirs(DATA_DIR, exist_ok=True)

wfdb.dl_database('mitdb', dl_dir=DATA_DIR)

In [ ]:
#probamos si los registros se bajaron bien. Accedemos a un elemento rdrecord de wfdb y lo guardamos en una nueva variable record. Lo mismo para las anotaciones, accedemos a un elemento rdann.
elemento = 100
record = wfdb.rdrecord(f'{DATA_DIR}/{elemento}')
anotacion = wfdb.rdann(f'{DATA_DIR}/{elemento}', 'atr')

In [ ]:
#Prints para verificar

print("Frecuencia de muestreo:", record.fs)
print("Derivaciones:", record.sig_name)
print("Duración en muestras:", record.sig_len)
print("Forma de la señal:", record.p_signal.shape)
print("Cantidad de latidos anotados:", len(anotacion.sample))
print("Primeros 10 símbolos de latido:", anotacion.symbol[:10])

#Todos los datos se acceden mediante los atributos de las respectivas funciones rdrecord y rdann

Paso 2) Filtrado por DII

In [ ]:
#Mapeo los registros que tienen derivaci'on II en MLII

registros_mlii = []
otros_registros = []

todos_los_registros = wfdb.get_record_list('mitdb')

for nom_reg in todos_los_registros:
  header = wfdb.rdheader(f'{DATA_DIR}/{nom_reg}')
  if header.sig_name[0] == 'MLII':
    registros_mlii.append(nom_reg)
  else:
    otros_registros.append(nom_reg)

In [ ]:
print(f"Num de registros con MLII en canal 1: {len(registros_mlii)}")
print(registros_mlii)
print(f"\n Num de registros SIN MLII en canal 1: {len(otros_registros)}")
print(otros_registros)

In [ ]:
#Trabajamos de aca en adelante solamente con registros_mlii.
#otros_registros se excluye.
#Esto es porque el sistema debe ser compatible con un ergometro single lead de DII.
#Por lo tanto a la CNN1D la entrenaremos con senales de DII.
#Aunque seguramente los registros tengan MLII en otro canal, por simplicidad simplemente los excluimos en principio.

Paso 3) Split de pacientes

In [ ]:
#Implementamos el Split de de Chazal et al.

Tomado de de Chazal et al. (2004): "The interpatient assessment model divides the MIT-BIH database into two groups of records: DS1 = {101, 106, 108, 109, 112, 114, 115, 116, 118, 119, 122, 124, 201, 203, 205, 207, 208, 209, 215, 220, 223, 230} and DS2 = {100, 103, 105, 111, 113, 117, 121, 123, 200, 202, 210, 212, 213, 214, 219, 221, 222, 228, 231, 232, 233, 234}. In addition, in order to explore the potential of this work, we divide DS1 (training set) into DS1_2 = {101, 106, 108, 109, 112, 114, 115, 116, 118, 119, 122}, DS1_3 = {124, 201, 203, 205, 207, 208, 209, 215, 220, 223, 230}. Detailed data sample distribution is shown in Supplementary Table S2. We use DS1, DS1_2 and DS1-3 to train classification models, and use DS2 to evaluate the training models."

In [ ]:
DS1 = ['101', '106', '108', '109', '112', '114', '115', '116', '118', '119',
       '122', '124', '201', '203', '205', '207', '208', '209', '215', '220',
       '223', '230']

DS2 = ['100', '103', '105', '111', '113', '117', '121', '123', '200', '202',
       '210', '212', '213', '214', '219', '221', '222', '228', '231', '232',
       '233', '234']

print(f"DS1 (train): {len(DS1)} registros")
print(f"DS2 (test): {len(DS2)} registros")
print(f"Total: {len(DS1) + len(DS2)} de 48 registros")

In [ ]:
#En nuestro caso, mantenemos el mismo split. Incorporamos 114 al flujo

In [ ]:
record_114 = wfdb.rdrecord(f'{DATA_DIR}/114')
idx = record.sig_name.index('MLII')
print(f"MLII está en la posición {idx} del registro 114")
print(f"Canales del registro: {record_114.sig_name}")

# Extraer la señal MLII correctamente, sea cual sea su posición
signal_mlii = record_114.p_signal[:, idx]
print(f"Forma de la señal MLII extraída: {signal_mlii.shape}")

In [ ]:
# Actualizamos la exclusión: solo 102 y 104 quedan afuera
mlii_records = []
excluded_records = []

for rec_name in todos_los_registros:
    header = wfdb.rdheader(f'{DATA_DIR}/{rec_name}')
    if 'MLII' in header.sig_name:
        mlii_records.append(rec_name)
    else:
        excluded_records.append(rec_name)

print(f"Registros con MLII (en cualquier posición): {len(mlii_records)}")
print(f"Registros excluidos (sin MLII en ningún canal): {excluded_records}")